# Feature Track 1: Evaluation & Validation

---

Shipping a RAG system without systematic evaluation is like navigating without instruments. The pipeline may *seem* to work on the queries you tested by hand, but you have no way to know where it breaks, how often, or whether a change you made helped or hurt.

**Evaluation closes the feedback loop:**

```
Change a parameter  ──►  Measure quantitatively  ──►  Decide based on data
```

**Prerequisite:** `conversational-toolkit` and `backend` must be installed in editable mode. Set `OPENAI_API_KEY` in your environment.

---

## Why Systematic Evaluation?
Suppose you change the chunk size from 800 to 400 characters. Did that help? How would you know?

Without metrics you are forced to manually re-read answers for a handful of test queries and guess. With metrics you run the evaluation suite and get a number -> a number you can track across changes and use to justify decisions.

### A Concrete Example from Feature 0
In Feature 0 we saw that the baseline RAG sometimes:
- Described the **Lara Pallet** as if it exists (it doesn't)
- Cited the **outdated 2021 GWP figure** even though a newer, verified EPD supersedes it
- Reported the tesa **68% CO₂ reduction** without flagging it as unverified

These are not edge cases -> they are exactly the queries that matter for compliance.
How often does this happen? After every change to the pipeline, you need an answer.

### The Four Questions Evaluation Answers
| Question | Why it matters |
|---|---|
| **Is the retriever finding the right chunks?** | A perfect LLM cannot fix wrong retrieval |
| **Is the LLM hallucinating?** | A fabricated GWP figure can be shared with clients |
| **Is the answer complete?** | Missing "this is unverified" can mislead a user |
| **Did my change help?** | Without a baseline metric you cannot tell improvement from regression |

---

### The RAG Pipeline

Each arrow is a potential failure point. Evaluation targets a specific stage so you can isolate *where* the problem is.


**Ingestion** (run once):
```
  Documents  ──►  [1] Chunker  ──►  [2] Embedder  ──►  [3] Vector DB
```

**Querying** (every user question):
```
  User query  ──►  [2] Embedder  ──►  [3] Retriever  ──►  Top-k Chunks
                                                                 │
                                                          [4] LLM + Prompt
                                                                 │
                                                          Answer + Sources
```

| Step | What it does | If it fails |
|---|---|---|
| [1] Chunking | Split documents into searchable units | Context split mid-fact; tables broken; information lost |
| [2] Embedding | Convert text to vectors | Wrong chunks returned despite a matching query |
| [3] Vector search | Find most similar chunks to query | Relevant chunks not returned |
| [4] Generation | LLM answers using retrieved context | Hallucination; ignores context; incomplete answer |

---

## RAGAS

[RAGAS](https://docs.ragas.io) (*Retrieval Augmented Generation Assessment*) is an open-source Python library for evaluating RAG pipelines.

#### How it works internally
Rather than asking a judge LLM "rate this answer 0–5", RAGAS decomposes the answer into individual atomic claims:

```
Answer: "The Logypal 1 GWP is 3.2 kg CO₂e, verified by Bureau Veritas."

  Claim 1: "GWP is 3.2 kg CO₂e"          -> supported by context?  ✓
  Claim 2: "verified by Bureau Veritas"  -> supported by context?  ✓

  Faithfulness = 2 supported / 2 total = 1.0
```

This catches partial hallucination, e.g. a correct figure with a fabricated verifier name.

#### Metrics overview

| Metric | Ground truth needed? | What it catches |
|---|---|---|
| `Faithfulness` | No | Claims not supported by the retrieved context |
| `AnswerRelevancy` | No | Off-topic or evasive answers |
| `ContextPrecision` | **Yes** | Irrelevant chunks ranked above relevant ones |
| `ContextRecall` | **Yes** | Facts needed to answer that are missing from the retrieved context |

The first two metrics can be run on any query with zero labelling effort. The last two require a **test set**: a curated list of `(query, reference answer)` pairs, which this notebook imports from `feature1_evaluation.py`.

#### Strengths and weaknesses
- Standardised, reproducible metrics widely used in industry
- Needs a capable judge LLM (defaults to OpenAI): typically 3+ LLM calls per sample per metric
- LLM judge has its own biases; metrics are proxies, not absolute ground truth

---

### RAGAS Setup

**Prerequisites:** `conversational-toolkit` and `backend` installed in editable mode.

RAGAS uses OpenAI as its judge LLM by default. Set `OPENAI_API_KEY` in your environment.

In [ ]:
import os
import pathlib
import warnings

from langchain_openai import ChatOpenAI, OpenAIEmbeddings as LangChainOpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper  # type: ignore[import-untyped]
from ragas.metrics import (  # type: ignore[attr-defined]
    Faithfulness as RagasFaithfulness,
    AnswerRelevancy as RagasAnswerRelevancy,
)
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import (
    OpenAIEmbeddings,
)
from conversational_toolkit.evaluation import Evaluator
from conversational_toolkit.evaluation.adapters import evaluate_with_ragas
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    EMBEDDING_MODEL,
    VS_PATH,
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    build_llm,
    build_agent,
    load_chunks,
    build_vector_store,
)

warnings.filterwarnings("ignore", category=DeprecationWarning)

_secret_path = pathlib.Path("/secrets/OPENAI_API_KEY")
if "OPENAI_API_KEY" not in os.environ and _secret_path.exists():
    os.environ["OPENAI_API_KEY"] = _secret_path.read_text().strip()

BACKEND = "openai"  # "ollama" or "openai"
# Note: RAGAS uses OpenAI for its judge LLM regardless of BACKEND above.

In [ ]:
FORCE_REBUILD = False  # set True to re-embed from scratch

if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks()
    vs = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vs = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vs.collection.count()} chunks)"
    )

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

llm = build_llm(backend=BACKEND)
agent = build_agent(
    vector_store=vs,
    embedding_model=embedding_model,
    llm=llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
)

In [ ]:
ragas_embeddings = LangChainOpenAIEmbeddings(model="text-embedding-3-small")

# RAGAS defaults to max_tokens=3072 for its judge LLM. Long answers with many atomic claims overflow this limit mid-JSON, causing "output is incomplete" errors.
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", max_completion_tokens=8192)
)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Vector store: {VS_PATH}")
print(f"RAG agent LLM: {BACKEND}")
print("RAGAS judge LLM: gpt-4o-mini (OpenAI)")
print("Setup complete.")

---

## Part 1: Generation Quality (No Ground Truth Required)

**Faithfulness** and **AnswerRelevancy** only need the query and the system's response; no reference answer required. We run them on the full test set from `feature1_evaluation.py`, and reuse the same samples in Part 2.

*Can take a few minutes: RAGAS makes multiple judge LLM calls per sample.*

In [ ]:
from sme_kt_zh_collaboration_rag.feature1_evaluation import EVALUATION_QUERIES

queries = [q["query"] for q in EVALUATION_QUERIES]
gt_answers = [q["ground_truth_answer"] for q in EVALUATION_QUERIES]

print(
    f"Building {len(queries)} evaluation samples (runs the RAG agent once per query)..."
)
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")

# Ground truth answers are stored in the samples for Part 2 (ContextPrecision, ContextRecall).
samples = await Evaluator.build_samples_from_agent(
    agent=agent,
    queries=queries,
    ground_truth_answers=gt_answers,
)
print(f"\nDone. {len(samples)} samples built.\n")

print("Running RAGAS Part 1: Faithfulness + AnswerRelevancy\n")

report_basic = evaluate_with_ragas(
    samples=samples,
    metrics=[
        RagasFaithfulness(),  # type: ignore[call-arg]
        RagasAnswerRelevancy(strictness=1),  # type: ignore[call-arg]
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("─" * 30)
print(f"Samples evaluated: {report_basic.num_samples}")
print("─" * 30)
for metric_name, score in report_basic.summary().items():
    print(f"{metric_name:<22}  {score:.3f}")
print("─" * 30)

In [ ]:
# Per-sample breakdown: find which queries score best / worst so we know where to focus improvement efforts.

import math

f_result = next(
    (r for r in report_basic.results if "faithfulness" in r.metric_name.lower()), None
)
a_result = next(
    (
        r
        for r in report_basic.results
        if "relevancy" in r.metric_name.lower() or "relevance" in r.metric_name.lower()
    ),
    None,
)

f_scores: list[float] = (f_result.per_sample_scores if f_result else None) or []
a_scores: list[float] = (a_result.per_sample_scores if a_result else None) or []


def fmt(v: float) -> str:
    return "  N/A" if math.isnan(v) else f"{v:>5.2f}"


print("Per-sample scores (F = Faithfulness, A = AnswerRelevancy)\n")
print(f"{'#':<3} {'F':>4} {'A':>5} {'query':>7} {'response':>45}")
print("─" * 100)
for i, (sample, f, a) in enumerate(zip(samples, f_scores, a_scores), 1):
    q = sample.query[:40] + ".." if len(sample.query) > 40 else sample.query
    r = (
        (sample.answer[:40] or "") + ".."
        if len(sample.answer or "") > 60
        else (sample.answer or "")
    )
    print(f"{i:<3} {fmt(f)} {fmt(a)} {q:<40} {r}")

---

## Part 2: Retrieval Quality (Ground Truth Required)

Faithfulness and AnswerRelevancy tell us about generation quality, but say nothing about whether the retriever found the *right* chunks. For that we use the ground truth answers already stored in `samples`.

| Metric | Score = 1.0 means | Low score means |
|---|---|---|
| **ContextPrecision** | Every retrieved chunk is relevant and the most relevant are ranked first | Retriever returns noisy chunks or ranks them poorly |
| **ContextRecall** | Every fact needed to answer the question is present in the retrieved context | Retriever is missing chunks that contain key facts |

In [ ]:
from ragas.metrics import (  # type: ignore[attr-defined]
    ContextPrecision as RagasContextPrecision,
    ContextRecall as RagasContextRecall,
)

print("Running RAGAS Part 2: ContextPrecision + ContextRecall\n")

report_context = evaluate_with_ragas(
    samples=samples,
    metrics=[
        RagasContextPrecision(),  # type: ignore[call-arg]
        RagasContextRecall(),  # type: ignore[call-arg]
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print("─" * 36)
print(f"Samples evaluated: {report_context.num_samples}")
print("─" * 36)
for metric_name, score in report_context.summary().items():
    print(f"{metric_name:<28}  {score:.3f}")
print("─" * 36)

In [ ]:
# Per-sample breakdown: find which queries have retrieval gaps


def fmt(v: float) -> str:
    return "  N/A" if math.isnan(v) else f"{v:>5.2f}"


cp_result = next(
    (r for r in report_context.results if "precision" in r.metric_name.lower()), None
)
cr_result = next(
    (r for r in report_context.results if "recall" in r.metric_name.lower()), None
)

cp_scores: list[float] = (cp_result.per_sample_scores if cp_result else None) or []
cr_scores: list[float] = (cr_result.per_sample_scores if cr_result else None) or []

print("Per-sample context scores (CP = ContextPrecision, CR = ContextRecall)\n")
print(f"{'#':<3} {'CP':>5} {'CR':>5}  query")
print("─" * 90)
for i, (sample, cp, cr) in enumerate(zip(samples, cp_scores, cr_scores), 1):
    q = sample.query[:68] + "..." if len(sample.query) > 68 else sample.query
    print(f"{i:<3} {fmt(cp):>5} {fmt(cr):>5}  {q}")

---

## Summary

Four RAGAS metrics covering both generation and retrieval quality:

| Metric | Ground truth? | Tells you |
|---|---|---|
| `Faithfulness` | No | Are answers grounded in the retrieved context? |
| `AnswerRelevancy` | No | Are answers on-topic? |
| `ContextPrecision` | Yes | Are retrieved chunks relevant and well-ranked? |
| `ContextRecall` | Yes | Does the retrieved context contain all necessary facts? |

Low generation scores -> look at prompt or LLM. Low retrieval scores -> look at chunking, embedding, or `top_k`.